# Aurora Greater Horn Forecasting Axis

This notebook adds the second research axis: weather/climate forecasting foundation models. The locked scope is **Aurora Small** and **Aurora Large** on a **WeatherBench2 HRES T0 Greater Horn of Africa** subset.

The notebook is deliberately strict: it checks real data access and real Aurora package availability before any forecast experiment is run.

## 1. Imports

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'eval_harness').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from eval_harness.config import dataset_config_path, load_yaml, model_config_path
from eval_harness.forecasting import (
    check_aurora_readiness,
    open_forecasting_dataset,
    summarize_forecasting_dataset,
)
from eval_harness.forecasting.regions import first_existing_coord

## 2. Load Locked Configs

In [ ]:
dataset_name = 'weatherbench2_hres_t0_greater_horn'
model_names = ['aurora_small', 'aurora_large']

dataset_cfg = load_yaml(dataset_config_path(dataset_name))
model_cfgs = {name: load_yaml(model_config_path(name)) for name in model_names}

print(dataset_cfg['display_name'])
pd.DataFrame([
    {
        'model': cfg['display_name'],
        'family': cfg['model_family'],
        'estimated_parameters': cfg.get('parameter_count_estimate'),
        'role': cfg.get('selection_role'),
        'status': cfg.get('integration_status'),
    }
    for cfg in model_cfgs.values()
])

## 3. Readiness Gate

Set `WEATHERBENCH2_HRES_T0_URI` to a WeatherBench2/HRES T0 Zarr or NetCDF path before running the dataset cells. The Aurora cells also require the actual Aurora package/checkpoints in the active kernel.

In [ ]:
dataset_readiness = summarize_forecasting_dataset(dataset_cfg, split='val').as_dict()
model_readiness = [check_aurora_readiness(cfg).as_dict() for cfg in model_cfgs.values()]

display(pd.DataFrame([dataset_readiness]))
display(pd.DataFrame(model_readiness))

## 4. Open Greater Horn Subset

In [ ]:
try:
    ds = open_forecasting_dataset(dataset_cfg, split='val', subset=True)
    print(ds)
except Exception as exc:
    ds = None
    print(f'Dataset is not ready yet: {type(exc).__name__}: {exc}')

## 5. Data Exploration

This verifies the actual region subset before model runs: dimensions, available variables, and a first spatial slice.

In [ ]:
if ds is not None:
    display(pd.DataFrame({'dimension': list(ds.sizes.keys()), 'size': list(ds.sizes.values())}))
    display(pd.DataFrame({'variable': sorted(ds.data_vars)}).head(60))

In [ ]:
if ds is not None:
    candidate_variables = ['2m_temperature', '2t', 'temperature_2m']
    variable = next((name for name in candidate_variables if name in ds.data_vars), None)
    if variable is None:
        print('No 2m-temperature-like variable found. Update candidate_variables after inspecting ds.data_vars.')
    else:
        da = ds[variable].isel({dataset_cfg['data'].get('time_coord', 'time'): 0})
        while len(da.dims) > 2:
            extra_dim = next(dim for dim in da.dims if dim not in {'latitude', 'lat', 'longitude', 'lon'})
            da = da.isel({extra_dim: 0})
        ax = da.plot(figsize=(8, 5), cmap='viridis')
        plt.title(f'{dataset_cfg["region"]}: {variable}, first validation time', fontweight='bold')
        plt.show()

## 6. Persistence Baseline Check

This is not an Aurora result. It is a data/metric sanity check: use the current atmospheric state as the forecast for future lead times, then compute weighted RMSE/MAE over the Greater Horn subset.

In [ ]:
if ds is not None:
    command = (
        'python scripts/run_forecasting_persistence_baseline.py '
        '--dataset weatherbench2_hres_t0_greater_horn '
        '--split val '
        '--variables 2m_temperature,10m_u_component_of_wind,10m_v_component_of_wind '
        '--lead-times-hours 6,12,24,48 '
        '--max-times 16'
    )
    print(command)
else:
    print('Skipping persistence baseline until dataset access is configured.')

## 7. Aurora Gate

Run this before attempting real forecasts. It should show `package_available = True` for the active environment. If it does not, install/provide Aurora first.

In [ ]:
aurora_status = pd.DataFrame([check_aurora_readiness(cfg).as_dict() for cfg in model_cfgs.values()])
display(aurora_status)

if not aurora_status['package_available'].all():
    print('Aurora is not ready in this kernel. Do not run model forecasts yet.')
else:
    print('Aurora package is importable. Next step: wire package-specific constructors and checkpoint paths.')